# Build Embedding Dataset

Goal:
Convert selected audio samples into embeddings.

Input:
ESC50 audio

Output:
Embeddings + labels

Pipeline:

Audio
↓

YAMNet
↓

Embedding
↓

Training Dataset

In [39]:
from pathlib import Path

import pandas as pd

import numpy as np

import librosa

import tensorflow_hub as hub

from tqdm import tqdm

In [40]:
ROOT = Path("../data")

RAW = ROOT / "raw" / "esc50"

AUDIO = RAW / "audio"

META = RAW / "meta" / "esc50.csv"

PROCESSED = ROOT / "processed"

PROCESSED.mkdir(
    exist_ok=True
)

In [41]:
df = pd.read_csv(
    META
)

In [42]:
df.head()

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


## Select Aegis Sounds

In [43]:
TARGET = {
    "alarm": [
        "clock_alarm"
    ],

    "baby": [
        "crying_baby"
    ],

    "knock": [
        "door_wood_knock"
    ],

    "siren": [
        "siren"
    ]
}

In [44]:
selected = []

for values in TARGET.values():

    selected.extend(
        values
    )

In [45]:
subset = df[
    df["category"].isin(
        selected
    )
]

len(subset)

160

## Load Model

In [46]:
model = hub.load(
    "https://tfhub.dev/google/yamnet/1"
)

## Extract Embeddings

In [47]:
X = []

y = []

meta = []

In [48]:
for _, row in tqdm(
    subset.iterrows(),
    total=len(subset)
):

    path = AUDIO / row["filename"]

    waveform, sr = librosa.load(
        path,
        sr=16000
    )

    waveform = waveform.astype(
        np.float32
    )

    scores, embeddings, spec = model(
        waveform
    )

    emb = (
        embeddings
        .numpy()
        .mean(axis=0)
    )

    X.append(
        emb
    )

    y.append(
        row["category"]
    )

    meta.append(
        row["filename"]
    )

100%|██████████| 160/160 [00:04<00:00, 34.59it/s]


In [49]:
X = np.array(
    X
)

y = np.array(
    y
)

print(
    X.shape
)

print(
    y.shape
)

(160, 1024)
(160,)


## Save Processed Dataset

In [50]:
np.save(
    PROCESSED / "X.npy",
    X
)

np.save(
    PROCESSED / "y.npy",
    y
)

In [51]:
pd.DataFrame({

    "file": meta,

    "label": y

}).to_csv(

    PROCESSED /
    "metadata.csv",

    index=False
)

# Findings

Embedding extraction works.

Dataset generated.

Ready for training.